# Variables de texto con BERT / MiniLM en Google Colab

Notebook para correr `04_text_features.py` (embeddings) y `05_text_finetune.py` (fine-tuning) con GPU.

**Antes de empezar**
1. En Google Drive crea la carpeta `MyDrive/taller3` con esta estructura:
```
taller3/
├── 02_code/04_text_features.py
├── 02_code/05_text_finetune.py
└── 01_data/01_raw/train.csv, test.csv
```
2. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.
3. (Opcional) Secreto `HF_TOKEN` en el ícono 🔑 de la izquierda. No es obligatorio: los modelos son públicos.

Todos los resultados se guardan en Drive, así que si Colab se desconecta no se pierde lo que ya terminó.

## 1. Conectar Drive y revisar la GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/taller3"
!ls 02_code 01_data/01_raw
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Instalar librerías
`torch` y `transformers` ya vienen en Colab; solo falta `sentence-transformers`.

In [ ]:
!pip install -q sentence-transformers

## 3. Embeddings congelados + TF-IDF (`04_text_features.py`)
Genera `01_data/02_text/text_features_{train,test}.csv`. En GPU tarda unos minutos.

In [ ]:
!python 02_code/04_text_features.py

## 4. Fine-tuning: prueba con 1 fold
Primero un solo fold con cada modelo para comparar tiempo y error. Mira la línea `MAE fold espacial` al final de cada corrida: **gana el que tenga menor MAE**.

In [ ]:
!python 02_code/05_text_finetune.py --model minilm --folds 0

In [ ]:
!python 02_code/05_text_finetune.py --model beto --folds 0

## 5. Fine-tuning: los 5 folds con el modelo ganador
Cambia `minilm` por `beto` si BETO ganó. El fold 0 ya está hecho y se salta. Si Colab se desconecta, vuelve a correr las celdas 1, 2 y esta: retoma desde el fold que falte.

Al terminar crea `01_data/02_text/text_finetune_<modelo>_{train,test}.csv` y `03_outputs/02_text/metricas_finetune_<modelo>.csv`.

In [ ]:
!python 02_code/05_text_finetune.py --model minilm

## 6. Revisar resultados

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob('03_outputs/02_text/metricas*.csv')):
    print(f); print(pd.read_csv(f).to_string(index=False), '
')

## 7. Descargar
Descarga desde Drive estos archivos y cópialos a las mismas rutas del repositorio local:
- `01_data/02_text/text_features_train.csv`, `text_features_test.csv`
- `01_data/02_text/text_finetune_<modelo>_train.csv`, `text_finetune_<modelo>_test.csv`
- `03_outputs/02_text/*.csv`

No hace falta descargar `01_data/02_text/cache/` ni `01_data/02_text/finetune/` (archivos intermedios).